In [12]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import os

client = QdrantClient(
    url="http://localhost:6336",
    api_key="Fucking_password_1234567890",
)

# For Colab:
# from google.colab import userdata
# client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

encoder = SentenceTransformer("all-MiniLM-L6-v2")

/tmp/ipykernel_2819034/287409236.py:5: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 33300.96it/s]


In [13]:
my_dataset = [
    {
        "title": "Classic Beef Bourguignon",
        "description": """A rich, wine-braised beef stew from Burgundy, France. 
        Tender chunks of beef are slowly simmered with pearl onions, mushrooms, 
        and bacon in a deep red wine sauce. The long, slow cooking process 
        develops complex flavors and creates a luxurious, velvety texture. 
        Perfect for cold winter evenings when you want something hearty and 
        comforting. Traditionally served with crusty bread or creamy mashed 
        potatoes to soak up the incredible sauce.""",
        "cuisine": "French",
        "difficulty": "Intermediate",
        "time": "3 hours"
    },
    {
        "title": "Vegetarian Mushroom Risotto",
        "cuisine": "Italian",
        "difficulty": "Intermediate",
        "time": "3 hours",
        "description": """A creamy, comforting rice dish packed with a variety of fresh mushrooms. 
        The risotto is made with a rich, savory broth and finished with a generous amount of 
        parmesan cheese. Topped with fresh herbs and a drizzle of olive oil, this dish is 
        perfect for a cozy dinner or as a side to complement any main course. A delightful 
        vegetarian option that's both satisfying and elegant."""
    },
    {
        "title": "Spicy Thai Green Curry",
        "cuisine": "Thai",
        "difficulty": "Intermediate",
        "time": "3 hours",
        "description": """A fragrant and fiery curry that combines the bold flavors of Thailand. 
        Made with coconut milk, green chilies, and a blend of aromatic herbs, this curry 
        is both spicy and aromatic. Perfect for those who enjoy a kick of heat with their 
        meals, it's typically served with steamed rice and fresh cilantro."""
    },
    {
        "title": "Mediterranean Quinoa Salad",
        "cuisine": "Mediterranean",
        "difficulty": "Easy",
        "time": "20 minutes",
        "description": """A refreshing and healthy salad made with quinoa, fresh vegetables, and a zesty lemon-olive oil dressing. 
        Packed with protein and fiber, this salad is perfect for a light lunch or as a side dish. 
        It's naturally gluten-free and can be easily customized with your favorite ingredients."""
    },
    {
        "title": "Classic Beef Bourguignon",
        "description": """A rich, wine-braised beef stew from Burgundy, France. 
        Tender chunks of beef are slowly simmered with pearl onions, mushrooms, 
        and bacon in a deep red wine sauce. The long, slow cooking process 
        develops complex flavors and creates a luxurious, velvety texture. 
        Perfect for cold winter evenings when you want something hearty and 
        comforting. Traditionally served with crusty bread or creamy mashed 
        potatoes to soak up the incredible sauce.""",
        "cuisine": "French",
        "difficulty": "Intermediate",
        "time": "3 hours"
    }

    
    ]

In [14]:

def fixed_size_chunks(text, chunk_size=100, overlap=20):
    """Split text into fixed-size chunks with overlap"""
    words  = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        if chunk_words:  # Only add non-empty chunks
            chunks.append(' '.join(chunk_words))
    
    return chunks

def sentence_chunks(text, max_sentences=3):
    """Group sentences into chunks"""
    import re
    sentences: list[str | Any] = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk_sentences = sentences[i:i + max_sentences]
        if chunk_sentences:
            chunks.append('. '.join(chunk_sentences) + '.')
    
    return chunks

def paragraph_chunks(text):
    """Split by paragraphs or double line breaks"""
    chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
    return chunks if chunks else [text]  # Fallback to full text

In [15]:
collection_name = "day1_semantic_search"

if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

# Create a collection with three named vectors
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "fixed": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "sentence": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "paragraph": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
)

# Index fields for filtering (more on this on day 2)
client.create_payload_index(
    collection_name=collection_name,
    field_name="chunk_strategy",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

# Process and upload data
points = []
point_id = 0

for item in my_dataset:
    description = item["description"]

    # Process with each chunking strategy
    strategies = {
        "fixed": fixed_size_chunks(description),
        "sentence": sentence_chunks(description),
        "paragraph": paragraph_chunks(description),
    }

    for strategy_name, chunks in strategies.items():
        for chunk_idx, chunk in enumerate(chunks):
            # Create vectors for this chunk
            vectors = {strategy_name: encoder.encode(chunk).tolist()}

            points.append(
                models.PointStruct(
                    id=point_id,
                    vector=vectors,
                    payload={
                        **item,  # Include all original metadata
                        "chunk": chunk,
                        "chunk_strategy": strategy_name,
                        "chunk_index": chunk_idx,
                    },
                )
            )
            point_id += 1

client.upload_points(collection_name=collection_name, points=points)
print(f"Uploaded {len(points)} chunks across three strategies")

Uploaded 18 chunks across three strategies


In [16]:
def compare_search_results(query):
    """Compare search results across all chunking strategies"""
    print(f"Query: '{query}'\n")

    for strategy in ["fixed", "sentence", "paragraph"]:
        results = client.query_points(
            collection_name=collection_name,
            query=encoder.encode(query).tolist(),
            using=strategy,
            limit=3,
        )

        print(f"--- {strategy.upper()} CHUNKING ---")
        for i, point in enumerate(results.points, 1):
            print(f"{i}. {point.payload['title']}")
            print(f"   Score: {point.score:.3f}")
            print(f"   Chunk: {point.payload['chunk'][:80]}...")
        print()


# Test with domain-specific queries
test_queries = [
    "comfort food for winter",  # Adapt these to your domain
    "quick and easy weeknight dinner",
    "elegant dish for special occasions",
]

for query in test_queries:
    compare_search_results(query)

Query: 'comfort food for winter'

--- FIXED CHUNKING ---
1. Vegetarian Mushroom Risotto
   Score: 0.330
   Chunk: A creamy, comforting rice dish packed with a variety of fresh mushrooms. The ris...
2. Classic Beef Bourguignon
   Score: 0.270
   Chunk: A rich, wine-braised beef stew from Burgundy, France. Tender chunks of beef are ...
3. Classic Beef Bourguignon
   Score: 0.270
   Chunk: A rich, wine-braised beef stew from Burgundy, France. Tender chunks of beef are ...

--- SENTENCE CHUNKING ---
1. Classic Beef Bourguignon
   Score: 0.557
   Chunk: Perfect for cold winter evenings when you want something hearty and 
        com...
2. Classic Beef Bourguignon
   Score: 0.557
   Chunk: Perfect for cold winter evenings when you want something hearty and 
        com...
3. Vegetarian Mushroom Risotto
   Score: 0.345
   Chunk: A delightful 
        vegetarian option that's both satisfying and elegant....

--- PARAGRAPH CHUNKING ---
1. Vegetarian Mushroom Risotto
   Score: 0.330
   Chunk: A 

In [17]:
def analyze_chunking_effectiveness():
    """Analyze which chunking strategy works best for your domain"""

    print("CHUNKING STRATEGY ANALYSIS")
    print("=" * 40)

    # Get chunk statistics for each strategy
    for strategy in ["fixed", "sentence", "paragraph"]:
        # Count chunks per strategy
        results = client.scroll(
            collection_name=collection_name,
            scroll_filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="chunk_strategy", match=models.MatchValue(value=strategy)
                    )
                ]
            ),
            limit=100,
        )

        chunks = results[0]
        chunk_sizes = [len(chunk.payload["chunk"]) for chunk in chunks]

        print(f"\n{strategy.upper()} STRATEGY:")
        print(f"  Total chunks: {len(chunks)}")
        print(f"  Avg chunk size: {sum(chunk_sizes)/len(chunk_sizes):.0f} chars")
        print(f"  Size range: {min(chunk_sizes)}-{max(chunk_sizes)} chars")


analyze_chunking_effectiveness()

CHUNKING STRATEGY ANALYSIS

FIXED STRATEGY:
  Total chunks: 5
  Avg chunk size: 366 chars
  Size range: 282-437 chars

SENTENCE STRATEGY:
  Total chunks: 8
  Avg chunk size: 243 chars
  Size range: 75-327 chars

PARAGRAPH STRATEGY:
  Total chunks: 5
  Avg chunk size: 404 chars
  Size range: 300-491 chars
